# Test Silver — MSPAS
Verifica las 3 tablas Silver de MSPAS sin escanear datos completos.
Usa `limit(10)` — no ejecuta `count()` ni agregaciones para no gastar créditos.

In [0]:
SILVER_SCHEMA = "stage_silver_ss2"

_TABLES = [
    "dbx_primeras_causas_de_morbilidad_2015_a_2025",
    "dbx_morbilidad_enfermedades_cronicas_2015_a_2025",
    "dbx_morbilidad_grupo_materno_infantil_2012_a_2025",
]

## dbx_primeras_causas_de_morbilidad_2015_a_2025

In [0]:
df = spark.table(f"{SILVER_SCHEMA}.dbx_primeras_causas_de_morbilidad_2015_a_2025")
df.printSchema()

In [0]:
display(df.limit(10))

## dbx_morbilidad_enfermedades_cronicas_2015_a_2025

In [0]:
df = spark.table(f"{SILVER_SCHEMA}.dbx_morbilidad_enfermedades_cronicas_2015_a_2025")
df.printSchema()

In [0]:
display(df.limit(10))

## dbx_morbilidad_grupo_materno_infantil_2012_a_2025

In [0]:
df = spark.table(f"{SILVER_SCHEMA}.dbx_morbilidad_grupo_materno_infantil_2012_a_2025")
df.printSchema()

In [0]:
display(df.limit(10))

## Chequeos rápidos de calidad (sin full scan)
Lee solo las primeras 1000 filas para verificar transformaciones clave.

In [0]:
from pyspark.sql import functions as F

for t in _TABLES:
    sample = spark.table(f"{SILVER_SCHEMA}.{t}").limit(1000)

    anio_nulls     = sample.filter(F.col("anio").isNull()).count()
    casos_nulls    = sample.filter(F.col("casos_total").isNull()).count()
    depto_nulls    = sample.filter(F.col("departamento_oficial").isNull()).count()
    # cie_10_norm no debe tener ':' ni espacios residuales (normalize_cie10 los elimina)
    cie_sucio      = sample.filter(F.col("cie_10_norm").rlike(r'[: ]')).count()
    cie_raw_sample = sample.select("cie_10", "cie_10_norm").limit(3).collect()

    print(f"\n{'─'*60}")
    print(f"Tabla : {t}")
    print(f"  anio nulls       : {anio_nulls}")
    print(f"  casos_total nulls: {casos_nulls}")
    print(f"  depto sin match  : {depto_nulls}")
    print(f"  cie_10_norm sucio: {cie_sucio}  (debe ser 0)")
    print(f"  CIE-10 muestra   : {[(r.cie_10, r.cie_10_norm) for r in cie_raw_sample]}")